In [16]:
"""
Paper-inspired Local-Ratio FAS + (heavier-first) add-back + fast edge-id graph
WITH practical parallelism:
  (1) Parallel DIMACS parsing + aggregation (by file byte ranges)
  (2) Parallel forward/backward evaluation
  (3) Optional parallel indegree computation for topo (usually not needed)

Notes:
- Phase 1 (cycle -> reduce) is inherently sequential (global mutable state).
- The parallelism here targets the big, easy wins: I/O + aggregation + evaluation.

Author: (you + ChatGPT)
"""

import os
import time
import heapq
import random
import pandas as pd
import multiprocessing as mp
from collections import defaultdict


# ============================================================
# 1) Parallel DIMACS reader (aggregates parallel arcs)
# ============================================================

def _parse_dimacs_range(args):
    """
    Worker: parse a file byte range [start, end), aggregate (u,v)->sum_w and node_ids.
    """
    file_path, start, end = args
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "rb") as f:
        f.seek(start)

        # If not starting at 0, discard partial line
        if start > 0:
            f.readline()

        while True:
            pos = f.tell()
            if pos >= end:
                break

            raw = f.readline()
            if not raw:
                break

            line = raw.strip()
            if not line:
                continue

            # We only care about lines starting with 'a'
            # Also skip 'c' and 'p'
            c0 = line[:1]
            if c0 in (b"c", b"p"):
                continue
            if c0 != b"a":
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            # parts: a u v w ...
            u = parts[1].decode("utf-8", errors="ignore")
            v = parts[2].decode("utf-8", errors="ignore")
            try:
                w = float(parts[3])
            except Exception:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    return agg, node_ids


def read_graph_dimacs_agg_parallel(file_path, n_procs=None):
    """
    Parallel DIMACS parsing with aggregation:
      - Aggregates parallel arcs (u,v) by summing weights
      - Deterministic node mapping (sorted node ids)
      - Deterministic edge order (sorted by (u_idx, v_idx))

    Returns:
      edges_indexed: list[(u_idx, v_idx, w_sum)]
      node_to_index: dict[node_id_str -> int]
      index_to_node: dict[int -> node_id_str]
    """
    if n_procs is None:
        n_procs = max(1, os.cpu_count() or 1)

    fsize = os.path.getsize(file_path)
    # Partition file into n_procs ranges
    step = max(1, fsize // n_procs)
    ranges = []
    start = 0
    for i in range(n_procs):
        end = fsize if i == n_procs - 1 else min(fsize, start + step)
        ranges.append((file_path, start, end))
        start = end

    # Parse in parallel
    with mp.Pool(processes=n_procs) as pool:
        results = pool.map(_parse_dimacs_range, ranges)

    # Merge
    agg = defaultdict(float)
    node_ids = set()
    for local_agg, local_nodes in results:
        node_ids.update(local_nodes)
        for k, w in local_agg.items():
            agg[k] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node


# ============================================================
# 2) Fast graph representation: edge IDs + active flags
# ============================================================

def build_eid_graph(edges_indexed, n_nodes, tol=1e-12):
    """
    Converts edges_indexed into edge-id arrays for speed.
    Returns:
      U, V: list[int] endpoints per edge id
      W0:  list[float] original weights
      W:   list[float] mutable reduced weights
      active: bytearray (1 if W>tol else 0)
      adj: list[list[int]] adjacency lists of edge IDs (outgoing)
    """
    m = len(edges_indexed)
    U = [0] * m
    V = [0] * m
    W0 = [0.0] * m
    W = [0.0] * m
    active = bytearray(m)
    adj = [[] for _ in range(n_nodes)]

    for eid, (u, v, w) in enumerate(edges_indexed):
        U[eid] = u
        V[eid] = v
        wf = float(w)
        W0[eid] = wf
        W[eid] = wf
        if wf > tol:
            active[eid] = 1
        adj[u].append(eid)

    # edges_indexed sorted => each adj[u] deterministic already
    return U, V, W0, W, active, adj


# ============================================================
# 3) Faster cycle finding (edge-id adjacency)
#    Resets only visited nodes (no O(n) clear each call)
# ============================================================

def find_any_cycle_eids(n_nodes, adj, V, active):
    """
    Finds any directed cycle in the active graph.
    Returns: list of edge IDs forming a directed cycle, or None if acyclic.
    """
    state = bytearray(n_nodes)        # 0=unvisited, 1=visiting, 2=done for THIS call
    parent = [-1] * n_nodes
    parent_eid = [-1] * n_nodes
    next_ptr = [0] * n_nodes
    touched = []

    def reset():
        for x in touched:
            state[x] = 0
            parent[x] = -1
            parent_eid[x] = -1
            next_ptr[x] = 0
        touched.clear()

    for s in range(n_nodes):
        if state[s] != 0:
            continue

        stack = [s]
        state[s] = 1
        touched.append(s)

        while stack:
            u = stack[-1]
            i = next_ptr[u]

            out = adj[u]
            # skip inactive edges
            while i < len(out) and active[out[i]] == 0:
                i += 1
            next_ptr[u] = i

            if i >= len(out):
                state[u] = 2
                stack.pop()
                continue

            eid = out[i]
            v = V[eid]
            next_ptr[u] = i + 1

            if state[v] == 0:
                parent[v] = u
                parent_eid[v] = eid
                state[v] = 1
                touched.append(v)
                stack.append(v)
            elif state[v] == 1:
                # back-edge => cycle; reconstruct as edge IDs
                cycle = [eid]
                cur = u
                while cur != v:
                    pe = parent_eid[cur]
                    if pe == -1:
                        break
                    cycle.append(pe)
                    cur = parent[cur]
                    if cur == -1:
                        break

                if cur == v:
                    cycle.reverse()
                    reset()
                    return cycle

        # continue with next start node

    reset()
    return None


# ============================================================
# 4) Topological order with optional quality improvement
# ============================================================

def topo_order_active(n_nodes, adj, V, active, priority_score=None):
    """
    Kahn topo sort on active graph.
    priority_score:
      - None: deterministic min-node tie break
      - list[float]: higher score wins among indeg-0 nodes (ties by node id)

    Returns:
      order: list of nodes
      rank:  list[int] rank[node]
    """
    indeg = [0] * n_nodes
    for u in range(n_nodes):
        for eid in adj[u]:
            if active[eid]:
                indeg[V[eid]] += 1

    heap = []
    if priority_score is None:
        for i in range(n_nodes):
            if indeg[i] == 0:
                heapq.heappush(heap, i)
        pop = lambda: heapq.heappop(heap)
        push = lambda x: heapq.heappush(heap, x)
    else:
        # max-score => use (-score, node)
        for i in range(n_nodes):
            if indeg[i] == 0:
                heapq.heappush(heap, (-priority_score[i], i))
        pop = lambda: heapq.heappop(heap)[1]
        push = lambda x: heapq.heappush(heap, (-priority_score[x], x))

    order = []
    while heap:
        u = pop()
        order.append(u)
        for eid in adj[u]:
            if not active[eid]:
                continue
            v = V[eid]
            indeg[v] -= 1
            if indeg[v] == 0:
                push(v)

    if len(order) != n_nodes:
        raise RuntimeError("Topological sort failed: active graph is not acyclic (unexpected).")

    rank = [0] * n_nodes
    for r, node in enumerate(order):
        rank[node] = r
    return order, rank


def compute_static_priority_scores(n_nodes, U, V, W0):
    """
    Cheap topo tie-break heuristic (quality):
      score[node] = total_out_weight - total_in_weight (original weights)
    Used only to break ties among indegree-0 nodes in topo ordering.
    """
    outw = [0.0] * n_nodes
    inw = [0.0] * n_nodes
    for eid in range(len(U)):
        u = U[eid]
        v = V[eid]
        w = W0[eid]
        outw[u] += w
        inw[v] += w
    return [outw[i] - inw[i] for i in range(n_nodes)]


# ============================================================
# 5) Reachability checker for add-back
# ============================================================

def make_reachability_checker(n_nodes, adj, V, active):
    visited = [0] * n_nodes
    stamp = 0

    def reachable(src, target, rank, rank_limit):
        nonlocal stamp
        stamp += 1
        st = stamp

        if src == target:
            return True
        if rank[src] > rank_limit:
            return False

        stack = [src]
        visited[src] = st

        while stack:
            x = stack.pop()
            for eid in adj[x]:
                if not active[eid]:
                    continue
                y = V[eid]
                if rank[y] > rank_limit:
                    continue
                if y == target:
                    return True
                if visited[y] != st:
                    visited[y] = st
                    stack.append(y)
        return False

    return reachable


# ============================================================
# 6) Local-ratio FAS with practical speedups + better add-back
# ============================================================

def local_ratio_fas_fast(edges_indexed, n_nodes, tol=1e-12, topo_priority=False):
    """
    Improvements:
      - no adjacency rebuild
      - cycle finder resets only touched nodes
      - add-back heavy edges first (by original weight)
      - add-back: O(1) accept if rank[u] < rank[v]
      - otherwise: reachability v->u with topo interval pruning
      - optional better topo tie-break (quality): static out-in score

    Returns:
      removed_eids: set[int]
      U,V,W0: arrays
      active, adj: final DAG
    """
    U, V, W0, W, active, adj = build_eid_graph(edges_indexed, n_nodes, tol=tol)
    removed_eids = set()

    # ---- Phase 1: local-ratio cycle reductions (sequential) ----
    while True:
        cyc = find_any_cycle_eids(n_nodes, adj, V, active)
        if cyc is None:
            break

        eps = None
        for eid in cyc:
            w = W[eid]
            if eps is None or w < eps:
                eps = w

        if eps is None or eps <= tol:
            eid0 = cyc[0]
            if active[eid0]:
                active[eid0] = 0
                W[eid0] = 0.0
                removed_eids.add(eid0)
            continue

        for eid in cyc:
            new_w = W[eid] - eps
            W[eid] = new_w
            if new_w <= tol and active[eid]:
                active[eid] = 0
                W[eid] = 0.0
                removed_eids.add(eid)

    # ---- Phase 2: add-back (heavy first) ----
    removed_list = sorted(list(removed_eids), key=lambda eid: (-W0[eid], U[eid], V[eid]))

    priority_score = compute_static_priority_scores(n_nodes, U, V, W0) if topo_priority else None
    _, rank = topo_order_active(n_nodes, adj, V, active, priority_score=priority_score)
    reachable = make_reachability_checker(n_nodes, adj, V, active)

    for eid in removed_list:
        u = U[eid]
        v = V[eid]

        # Fast accept if respects current topo order
        if rank[u] < rank[v]:
            active[eid] = 1
            removed_eids.discard(eid)
            continue

        # Otherwise cycle iff v reaches u
        if not reachable(v, u, rank=rank, rank_limit=rank[u]):
            active[eid] = 1
            removed_eids.discard(eid)
            # Since we accepted a "backward" edge w.r.t old rank, refresh rank.
            _, rank = topo_order_active(n_nodes, adj, V, active, priority_score=priority_score)

    return removed_eids, U, V, W0, active, adj


# ============================================================
# 7) Parallel forward/backward evaluation
# ============================================================

def _fw_bw_worker(args):
    edges_slice, scores = args
    total = 0.0
    fw = 0.0
    for u, v, w in edges_slice:
        total += w
        if scores[u] < scores[v]:
            fw += w
    return total, fw


def compute_forward_backward_parallel(edges_indexed, scores, n_procs=None):
    if n_procs is None:
        n_procs = max(1, os.cpu_count() or 1)

    m = len(edges_indexed)
    if m == 0:
        return 0.0, 0.0, 0.0

    # chunk edges
    chunk = max(1, m // n_procs)
    slices = []
    for i in range(n_procs):
        a = i * chunk
        b = m if i == n_procs - 1 else min(m, (i + 1) * chunk)
        if a < b:
            slices.append(edges_indexed[a:b])

    with mp.Pool(processes=n_procs) as pool:
        parts = pool.map(_fw_bw_worker, [(sl, scores) for sl in slices])

    total = sum(t for t, _ in parts)
    fw = sum(f for _, f in parts)
    bw = total - fw
    return total, fw, bw


# ============================================================
# 8) End-to-end: DIMACS -> paper FAS -> topo ranking CSV
# ============================================================

def paper_fas_ranking_from_dimacs_fast_parallel(
    dimacs_path,
    output_ranking_csv_path,
    tol=1e-12,
    parse_procs=None,
    eval_procs=None,
    topo_priority=False
):
    """
    - Parallel parsing + aggregation
    - Fast local-ratio + add-back
    - Optional topo tie-break heuristic for better ranking
    - Returns scores compatible with your FW/BW checker.
    """
    edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg_parallel(dimacs_path, n_procs=parse_procs)
    n = len(node_to_index)

    removed_eids, U, V, W0, active, adj = local_ratio_fas_fast(
        edges_indexed, n, tol=tol, topo_priority=topo_priority
    )

    priority_score = compute_static_priority_scores(n, U, V, W0) if topo_priority else None
    _, rank = topo_order_active(n, adj, V, active, priority_score=priority_score)

    # Write ranking: Node ID, Order
    rows = [{"Node ID": str(index_to_node[i]).strip(), "Order": int(rank[i])} for i in range(n)]
    rows.sort(key=lambda r: r["Order"])
    pd.DataFrame(rows).to_csv(output_ranking_csv_path, index=False)

    scores = {i: int(rank[i]) for i in range(n)}
    F_removed_pairs = {(U[eid], V[eid]) for eid in removed_eids}
    return edges_indexed, node_to_index, index_to_node, scores, F_removed_pairs


# ============================================================
# 9) Main
# ============================================================

if __name__ == "__main__":
    # IMPORTANT for multiprocessing on HPC:
    mp.set_start_method("fork", force=True)

    edge_file = "/mmfs1/home/sv96/Feedback-arc-set-paper/datasets/s38584.d"
    out_csv = edge_file.replace(".d", "") + "_paper_fas_ranking.csv"

    # Tune these:
    PARSE_PROCS = 24     # parallel file parsing/aggregation
    EVAL_PROCS  = 24     # parallel FW/BW evaluation
    TOPO_PRIORITY = True # better topo tie-break (quality), small overhead

    t0 = time.perf_counter()

    edges_indexed, node_to_index, index_to_node, scores, F_removed = paper_fas_ranking_from_dimacs_fast_parallel(
        dimacs_path=edge_file,
        output_ranking_csv_path=out_csv,
        tol=1e-12,
        parse_procs=PARSE_PROCS,
        eval_procs=EVAL_PROCS,
        topo_priority=TOPO_PRIORITY
    )

    # FW/BW (parallel)
    total_w, fw, bw = compute_forward_backward_parallel(edges_indexed, scores, n_procs=EVAL_PROCS)

    elapsed_sec = time.perf_counter() - t0

    print(f"✅ Wrote ranking: {out_csv}")
    print(f"Graph: n={len(node_to_index)} nodes, m={len(edges_indexed)} edges (after aggregation)")
    print(f"Total Weight: {total_w:.6f}")
    print(f"Forward Weight: {fw:.6f}")
    print(f"Backward Weight: {bw:.6f}")
    print(f"Forward Ratio: {fw/total_w:.6f}" if total_w > 0 else "Forward Ratio: N/A")
    print(f"Removed edges in minimal FAS (count): {len(F_removed)}")
    print(f"⏱️ Running time: {elapsed_sec:.3f} seconds ({elapsed_sec/60.0:.3f} minutes)")


✅ Wrote ranking: /mmfs1/home/sv96/Feedback-arc-set-paper/datasets/s38584_paper_fas_ranking.csv
Graph: n=20349 nodes, m=34558 edges (after aggregation)
Total Weight: 51943738.000000
Forward Weight: 51383388.000000
Backward Weight: 560350.000000
Forward Ratio: 0.989212
Removed edges in minimal FAS (count): 1205
⏱️ Running time: 11.190 seconds (0.187 minutes)
